# Coding Session #8

---

## Today's session

Today's session is structured to teach more complex tasks. We will explore:

- Demos + balance test tables
- Overall ATE
- Multiple ATEs/heterogeneity

### 0.1 Environment preparation

We begin by loading the libraries we’ll need. In Python, libraries are like toolkits: they extend the language with specialized functions.

- **pandas (pd)** is our main tool for working with tabular data. It introduces the DataFrame, which lets us manipulate datasets in a way that feels natural if you’ve used Excel or R.
- **NumPy (np)** provides the numerical backbone. It gives us arrays and fast mathematical functions, which pandas actually uses under the hood.
- **Seaborn (sns)** builds on top of Matplotlib to create a vast range of plots and visualization.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns


### 0.2 Reading Dataset

## Dataset Description

This dataset comes from a large-scale field experiment designed to study whether partisan get-out-the-vote (GOTV) phone calls can increase voter turnout in U.S. elections. The experiment was conducted in six states before the 2014 general election and included over 539,000 registered voters. Participants were randomly assigned to one of three treatment groups that received automated partisan phone calls (one call, three calls, or six calls) or to a control group that received no calls.

The random assignment allows researchers to estimate the causal effect of political campaign contact on voter turnout. Because the calls were explicitly partisan, the dataset is especially useful for campaigners. It allows us to answer the question of **How many robo calls should a campaign use to maximize voter turnout?**

### Citation

Kling, Daniel T., and Thomas Stratmann. 2023. *Large-Scale Evidence for the Effectiveness of Partisan GOTV Robo Calls*. *Journal of Experimental Political Science*. https://doi.org/10.1017/XPS.2022.16



#### Codebook


## Codebook

| Variable   | Type        | Description                                                                 |
|------------|-------------|-----------------------------------------------------------------------------|
| **age**    | Continuous  | Age of registered voter at time of experiment|
| **income** | Categorical | Estimated household income based on external vendor data.                   |
| **male**   | Binary      | Gender of voter.                                                            |
| **svh**    | Binary      | Indicator for single-voter household                                     |
| **gen2012**| Binary      | Indicator for having voted in the 2012 General Election.                    |
| **treatment** | Binary   | Indicator for assignment to any treatment condition.                        |
| **group**     | Binary   | Indicator for assignment to 1-call, 3-call, and 6-call treatment group.  |
| **voted** | Binary       | Indicator of having voted in 2014 General Election.                       |






In [ ]:
#Load "dime_candidates_2020_2024.csv" from remote repository
df_kling = pd.read_csv("https://raw.githubusercontent.com/albertostefanelli/DSPC_coding_sessions/refs/heads/master/codingsession08/data/kling_stratmann_subset.csv")

# Quick check
print(df_kling.dtypes)
print(df_kling.shape)



## 1. It the dataset representative of the US population?

In this section, we use the pre-treatment covariates to understand who is included in the experiment. Before estimating any treatment effects, it is essential to examine the demographic characteristics of the participants (such as age, gender, income, and past voting behavior).

This helps us determine whether the sample is representative of the broader voting population and ensures that the randomization process was applied to a population we can meaningfully generalize from.


In [ ]:
summary_cont = (
    df_kling[["treatment", "age","income","male", "svh", "gen2012"]]
    .agg(["count","mean","std", "median","min","max"])
    .T
)

summary_cont

There are several important observations from the pre-treatment summary statistics:

- **Missing Data:** There are missing values for age and income, which may need to be handled through imputation or listwise deletion depending on the analysis.
- **Treatment Assignment:** The mean of the `treatment` variable is 0.75, indicating that approximately 75% of the sample was assigned to a treatment group. This reflects an unbalanced experimental design rather than a 50/50 split.
- **Comparison to 2014 U.S. Census Benchmarks:**
  - **Median age:** 37.7 years — our sample is substantially older
  - **Median household income:** $53,657 — our sample is wealthier
  - **Male share of population:** 49.2% — closely aligned with the sample distribution.
  - **Single-person households:** 28% — higher than our measure of single-voter households
  - **National voter turnout in 2012:** 61.8% — our sample is higher

These comparisons show that the sample is not representative of the overall U.S. population but is instead more representative of active, registered voters, which is appropriate given the nature of the experiment.


## 2. Was the experiment properly implemented?

In a randomized experiment, a balance table compares pre-treatment covariates (e.g., age, income, gender, past vote) between the treatment and control groups. If randomization worked, these baseline characteristics should be similar on average across groups. Good balance supports the idea that any post-treatment differences in outcomes are caused by the treatment rather than pre-existing differences. Practically, we report group means, their differences, and simple checks (sometiems t-tests) to show that groups look alike before the intervention.


In [ ]:
def balance_table(df, treat_col, vars):
    """
    Creates a simple balance table that compares the mean of each variable
    between the treatment and control groups, and calculates the difference.
    """

    # Group the dataset by the treatment indicator (0 = control, 1 = treatment)
    group_means = df.groupby(treat_col)[vars].mean()

    # Transpose the table so that variables are rows and groups are columns
    group_means = group_means.T

    # Create a new column showing the difference in means (Treatment minus Control)
    group_means["diff"] = group_means[1] - group_means[0]

    # Step 4: Return the final balance table
    return group_means

In [ ]:
pretreat_vars = ["age", "income", "male", "svh", "gen2012"]
balance_table(df_kling, "treatment", pretreat_vars)

The balance table shows that the treatment and control groups are nearly identical across all pre-treatment characteristics, which indicates that random assignment worked properly. The average age and income differ only slightly, and the proportions of male participants, single-voter households, and past voters (from the 2010 election) are virtually the same in both groups. These very small differences suggest that neither group is systematically older, wealthier, or more likely to have voted in the past, which is exactly what we expect in a well-randomized experiment.

## 3. Calculate the ATE

To understand the impact of an experimental intervention, we compare the average outcome for individuals who received the treatment to those who did not. This difference in means is called the Average Treatment Effect (ATE). In a randomized experiment, this simple comparison provides an unbiased estimate of the causal effect because random assignment ensures that treatment and control groups are similar on all other factors. By calculating the average turnout in the treatment group and subtracting the average turnout in the control group, we directly measure the effect of the treatment on voter participation.


In [ ]:
def ate(df, treat_col, outcome_col):
    """
    Calculates the Average Treatment Effect (ATE)

    Parameters:
    df          : pandas DataFrame containing the data
    treat_col   : name of the treatment variable (1 = treated, 0 = control)
    outcome_col : name of the outcome variable (e.g., 'voted14')

    Prints the mean outcomes and treatment effect.
    """

    # Mean outcome for treated group
    av_treat = df.loc[df[treat_col] == 1, outcome_col].mean()

    # Mean outcome for control group
    av_control = df.loc[df[treat_col] == 0, outcome_col].mean()

    # Treatment effect (difference in means)
    ate = av_treat - av_control

    # Print results
    print(f"The average {outcome_col} in the treatment group is {round(av_treat, 3)}")
    print(f"The average {outcome_col} in the control group is {round(av_control, 3)}")
    print(f"The average treatment effect is {round(ate, 3)}")



Let's now calculate the overall ATE


In [ ]:
ate(df_kling, "treatment", "voted")

The treatment slightly increased voter turnout: about 49.2% of those who received the treatment voted, compared to 48.9% in the control group. This results in an average treatment effect of 0.3 percentage points.

Remember that our original research question asks: How many robo calls should a campaign use to maximize voter turnout? To answer this, we need to compare turnout across different treatment arms. We will start by checking how many respondents were assigned to each group. Then, we will create separate dummy variables to compare the control group against each treatment arm (1 call, 3 calls, and 6 calls) individually.

In [ ]:
# Check the number of respondents in each experimental group
df_kling['group'].value_counts().sort_index()

In [ ]:
# Start by setting all values to NaN
df_kling['control_vs_t1'] = np.nan
df_kling['control_vs_t3'] = np.nan
df_kling['control_vs_t6'] = np.nan

# Assign 1 to treatment group, 0 to control
df_kling.loc[df_kling['group'] == "T1: 1 Call", 'control_vs_t1'] = 1
df_kling.loc[df_kling['group'] == "Control (No Call)", 'control_vs_t1'] = 0

df_kling.loc[df_kling['group'] == "T3: 3 Calls", 'control_vs_t3'] = 1
df_kling.loc[df_kling['group'] == "Control (No Call)", 'control_vs_t3'] = 0

df_kling.loc[df_kling['group'] == "T6: 6 Calls", 'control_vs_t6'] = 1
df_kling.loc[df_kling['group'] == "Control (No Call)", 'control_vs_t6'] = 0

In [ ]:
ate(df_kling, "control_vs_t1", "voted")
ate(df_kling, "control_vs_t3", "voted")
ate(df_kling, "control_vs_t6", "voted")

## 4. Permutation test and p-values

We find effects that are around half a percentage point. At first glance, this may seem small and one might be tempted to dismiss these results as insignificant. However, increasing turnout by even half a percentage point through robocalls is meaningful, given how inexpensive they are to deploy at scale. To assess whether this effect is statistically significant, we will conduct a permutation test.

In [ ]:
# Let's first create a costum funciton to calculate the ATE
def calculate_ate(data, treatment, outcome):
    # Calculate the average treatment effect (ATE)
    return data.groupby(treatment)[outcome].mean().diff().iloc[-1]

# Let's check our funciton worked.
observed_ate = calculate_ate(df_kling, "control_vs_t1", "voted")
print('Observed ATE:', round(observed_ate, 3))

Now let’s take a look at the `permutation_test` function we saw in class. There are a few key ideas to keep in mind:

---
- Compared to last time We now **calculate a p-value**.
  - **What is a p-value?**  
    It tells us the probability of observing a treatment effect as large as (or larger than) the one we found, **just by chance**, if the treatment actually had *no effect at all*.
  - **One-sided vs. Two-sided p-values**:  
    - *One-sided test*: looks for an effect in **one direction only** (e.g., treatment *increases* turnout).
    - *Two-sided test*: checks for effects in **either direction** (treatment could increase *or* decrease turnout).
---
We are testing the **sharp null hypothesis**:

> **Treatment has zero effect on every individual.**

Under this assumption:

- **Outcomes are fixed**  
  We assume each person's outcome would be the same whether or not they received treatment.
- **We shuffle treatment assignments**  
  This simulates all the different ways the experiment *could have been randomized*. Each shuffle gives us a possible treatment effect we’d expect **just from chance**.
- **Why we *don’t* shuffle outcomes**  
  Shuffling outcomes would pretend that outcomes are random and unrelated to individuals. This ignores our actual experimental design and would not help us determine whether the treatment caused any change.



In [ ]:
# Function to perform the permutation test
def permutation_test(data, treatment, outcome, n_permutations=1000):
    observed_ate = calculate_ate(data, treatment, outcome)
    permuted_ates = []

    # Run the permutation test n_permutations times
    for _ in range(n_permutations):
        # Shuffle the treatment column
        shuffled_treat = np.random.permutation(data[treatment])

        # Create a copy of the data with the shuffled treatment assignments
        permuted_data = data.copy()
        permuted_data[treatment] = shuffled_treat

        # Calculate the ATE for each permutation.
        permuted_ate = calculate_ate(permuted_data, treatment, outcome)
        permuted_ates.append(permuted_ate)

    # Calculate the p-value as the proportion of permuted ATEs that are
    # as "extreme" as the observed ATE
    p_value = np.mean(np.abs(permuted_ates) >= np.abs(observed_ate))

    return p_value

In [ ]:
# Perform the permutation test and calculate the p-value
p_value_cvs1 = permutation_test(df_kling, "control_vs_t1", "voted", n_permutations=1000)
p_value_cvs3 = permutation_test(df_kling, "control_vs_t3", "voted", n_permutations=1000)
p_value_cvs6 = permutation_test(df_kling, "control_vs_t6", "voted", n_permutations=1000)

In [ ]:
print(p_value_cvs1)
print(p_value_cvs3)
print(p_value_cvs6)

### 5. [Home Work] Why is the permutation test result different from the paper’s result?

The original paper uses **regression with clustered standard errors** and leverages the extremely large sample size to detect very small effects. In contrast, a **permutation test**:

- Does *not* assume a regression model.
- Randomly reassigns treatment labels to create a distribution of the treatment effect under the null hypothesis of no effect.
- Is more conservative when the true effect is small and the signal-to-noise ratio is low.

In other words, **the permutation test evaluates whether the observed difference could plausibly arise by chance alone**, while the regression approach relies on **asymptotic (large sample) properties** to detect significance.


# Congratulations!

You are done with the coding session. Questions or suggestions? Email Alberto at alberto.stefanelli@yale.edu

In [ ]:
# Install requirements
!apt-get -qq update
!apt-get install -y pandoc texlive-xetex texlive-fonts-recommended texlive-plain-generic

from google.colab import drive, files

# Mount Google Drive
drive.mount('/content/drive')

# Ask for the notebook name
notebook_name = input(
    "Enter your notebook’s exact file name,\n"
    "exactly as shown in the top-left corner of the Colab page (next to the two yellow circle icons): "
)

# Build paths
input_path = f"/content/drive/MyDrive/Colab Notebooks/{notebook_name}"
output_path = input_path.replace(".ipynb", ".pdf")

# Convert to PDF
!jupyter nbconvert --to pdf "{input_path}"

# Download the PDF
files.download(output_path)